# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL from the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset collection.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs from the dataset metadata. All references use their `@id` property to ensure uniqueness and reproducibility.

In [ ]:
# List all available record sets and their fields using their @id
print("Record Sets (@id):")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id} | Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | Name: {field.name}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. For this exploration, we choose the primary tabular record set, using its `@id`, and field ids from the overview above.

In [ ]:
# Get a list of all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set into a dataframe
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records. Columns:", dataframes[record_set_id].columns.tolist())
    else:
        print(f"No records found for {record_set_id}.")

# For this example, select the first available tabular record set
main_record_set_id = None
for rid in record_set_ids:
    if rid in dataframes:
        main_record_set_id = rid
        break

if main_record_set_id is not None:
    print(f"\nExample data from RecordSet @id={main_record_set_id}:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record set with records found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping or aggregating data. We continue using the main record set identified above and reference fields only by their `@id`.

In [ ]:
import numpy as np

# Pick a representative numeric field for filtering and normalization
# Identify a numeric field (preferably protein abundance, MW, coverage) by its @id
numeric_field_id = None
for field in dataset.metadata.record_sets[0].fields:
    if field.data_type in ['Float', 'Integer', 'Number']:
        numeric_field_id = field.id
        print(f"Using numeric field: {numeric_field_id} ({field.name})")
        break
assert numeric_field_id is not None, "No numeric field found in the main record set."

df = dataframes[main_record_set_id]

# Try to convert the field to numeric, skipping NaNs
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Set a threshold, e.g., above median value
threshold = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)}")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field if available
group_field_id = None
for field in dataset.metadata.record_sets[0].fields:
    if field.data_type == 'Text' and field.id != numeric_field_id:
        group_field_id = field.id
        print(f"Grouping by field: {group_field_id} ({field.name})")
        break

if group_field_id is not None and group_field_id in filtered_df.columns:
    # Show mean of normalized field per group
    grouped_df = filtered_df.groupby(group_field_id)[f"{numeric_field_id}_normalized"].mean().reset_index()
    print(f"Mean normalized {numeric_field_id} per {group_field_id}:")
    display(grouped_df)
else:
    print("No appropriate group field for aggregation.")

## 5. Visualization
Visualize the distribution and relationships of the main numeric and categorical fields selected above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot by group if available
if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- This notebook demonstrates loading FAIR² Croissant-based datasets with `mlcroissant` using only `@id` references for record sets and fields.
- We explored record set and field structure programmatically and performed some simple data filtering, normalization, and visualization.
- For more advanced analysis, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/).

**Note:** Always reference dataset entities by their `@id` for portability and schema adherence.